# TRAFFIQ v2 Training Notebook

This notebook is converted from `train_dave_cv.py` and is ready for Colab/VS Code notebook execution.

Expected output: `[speed, direction]` with weighted loss.

In [7]:
# Optional: uncomment in Colab if dependencies are missing
!pip install tensorflow opencv-python scikit-learn matplotlib

In [8]:
# Colab: mount Google Drive and quickly check likely dataset folders
from pathlib import Path

IN_COLAB = Path('/content').exists()
if IN_COLAB:
    try:
        from google.colab import drive
        if not Path('/content/drive/MyDrive').exists():
            drive.mount('/content/drive')
        else:
            print('[Drive] Already mounted.')
    except Exception as e:
        print('[Drive] Mount skipped:', e)
else:
    print('[Drive] Not running in Colab.')

# Fast path checks only (no deep recursive scan)
candidates = [
    Path('/content/dataset_car'),
    Path('/content/dataset'),
    Path('/content/drive/MyDrive/dataset_car'),
    Path('/content/drive/MyDrive/dataset'),
    Path('/content/drive/MyDrive/Trafiic_car_autonomous/traffiq/dataset_car'),
    Path('/content/drive/MyDrive/Trafiic_car_autonomous/traffiq/dataset'),
    Path('/content/drive/MyDrive/traffiq/dataset_car'),
    Path('/content/drive/MyDrive/traffiq/dataset'),
]

print('\n[Discovery] Existing dataset directories:')
found = False
for p in candidates:
    if p.exists() and p.is_dir():
        print(' -', p)
        found = True

if not found:
    print(' - None of the common dataset paths were found.')
    print(' - Set DATA_DIR manually to your dataset folder path.')

In [9]:
# Colab speed prep: move dataset zip to local disk and point DATA_DIR to /content/dataset_car
import os
import shutil
import zipfile
from pathlib import Path

DRIVE_ZIP = Path('/content/drive/MyDrive/dataset_car.zip')
LOCAL_ZIP = Path('/content/dataset_car.zip')
LOCAL_DATA_DIR = Path('/content/dataset_car')

if Path('/content').exists():
    if LOCAL_DATA_DIR.exists() and any(LOCAL_DATA_DIR.iterdir()):
        print('[LocalData] Already extracted:', LOCAL_DATA_DIR)
    else:
        if not LOCAL_ZIP.exists():
            if DRIVE_ZIP.exists():
                print('[LocalData] Copying zip from Drive to local disk...')
                shutil.copy2(DRIVE_ZIP, LOCAL_ZIP)
            else:
                print('[LocalData] Zip not found on Drive:', DRIVE_ZIP)

        if LOCAL_ZIP.exists():
            print('[LocalData] Extracting zip to /content ...')
            with zipfile.ZipFile(LOCAL_ZIP, 'r') as zf:
                zf.extractall('/content')
            print('[LocalData] Extraction complete.')
        else:
            print('[LocalData] Skipping extraction: no zip file available.')

    if LOCAL_DATA_DIR.exists():
        os.environ['DATA_DIR'] = str(LOCAL_DATA_DIR)
        print('[LocalData] DATA_DIR forced to', os.environ['DATA_DIR'])
    else:
        print('[LocalData] /content/dataset_car not found yet. Using existing setup fallback.')
else:
    print('[LocalData] Not running in Colab; skipping local copy/extract.')

In [10]:
import os
import sys
import json
import glob
import random
import numpy as np
import cv2
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.model_selection import train_test_split

import tensorflow as tf
from tensorflow.keras import layers, callbacks, optimizers

In [11]:
# Robust setup for VS Code + Colab
import os
from pathlib import Path

def _find_project_root_for_cv_pipeline():
    cwd = Path.cwd().resolve()
    candidates = [cwd, *cwd.parents, Path('/content'), Path('/content/drive/MyDrive')]

    for base in candidates:
        if (base / 'scripts' / 'cv_pipeline.py').exists():
            return base
        if (base / 'traffiq' / 'scripts' / 'cv_pipeline.py').exists():
            return base / 'traffiq'

    # Deep recursive project search is optional because it can be slow on Drive.
    if os.environ.get('ENABLE_DEEP_PROJECT_SEARCH', '0') == '1':
        for root in [Path('/content/drive/MyDrive'), Path('/content'), cwd]:
            if not root.exists():
                continue
            try:
                for p in root.rglob('cv_pipeline.py'):
                    if p.parent.name == 'scripts':
                        return p.parent.parent
            except Exception:
                pass

    return None

def _is_valid_dataset_dir(p: Path) -> bool:
    if not p.exists() or not p.is_dir():
        return False
    # TRAFFIQ format
    if (p / 'labels.json').exists():
        return True
    # Donkey format: subfolders with catalog_*.catalog
    try:
        for sub in p.iterdir():
            if sub.is_dir() and list(sub.glob('catalog_*.catalog')):
                return True
    except Exception:
        pass
    return False

def _find_dataset_dir(project_root):
    candidates = []
    cwd = Path.cwd().resolve()

    if project_root is not None:
        candidates.extend([
            project_root / 'dataset_car',
            project_root / 'dataset',
            project_root.parent / 'dataset_car',
            project_root.parent / 'dataset',
        ])

    candidates.extend([
        cwd / 'dataset_car',
        cwd / 'dataset',
        Path('/content/dataset_car'),
        Path('/content/dataset'),
        Path('/content/drive/MyDrive/dataset_car'),
        Path('/content/drive/MyDrive/dataset'),
        Path('/content/drive/MyDrive/Trafiic_car_autonomous/traffiq/dataset_car'),
        Path('/content/drive/MyDrive/Trafiic_car_autonomous/traffiq/dataset'),
        Path('/content/drive/MyDrive/traffiq/dataset_car'),
        Path('/content/drive/MyDrive/traffiq/dataset'),
    ])

    for p in candidates:
        if _is_valid_dataset_dir(p):
            return p

    # Deep recursive dataset search is optional because it can take minutes on large drives.
    if os.environ.get('ENABLE_DEEP_DATASET_SEARCH', '0') == '1':
        for root in [Path('/content/drive/MyDrive'), Path('/content'), cwd]:
            if not root.exists():
                continue
            try:
                for p in root.rglob('*'):
                    if p.is_dir() and _is_valid_dataset_dir(p):
                        return p
            except Exception:
                pass

    return None

PROJECT_ROOT = _find_project_root_for_cv_pipeline()
if PROJECT_ROOT is not None and str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

try:
    from scripts.cv_pipeline import normalize_lighting, crop_frame, preprocess_for_cnn
    print('[Setup] Using scripts.cv_pipeline from:', PROJECT_ROOT)
except Exception as e:
    print('[Setup] scripts.cv_pipeline not found, using notebook fallback preprocessing.')
    print('[Setup] Import error:', e)

    def normalize_lighting(img):
        lab = cv2.cvtColor(img, cv2.COLOR_RGB2LAB)
        l, a, b = cv2.split(lab)
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
        l = clahe.apply(l)
        return cv2.cvtColor(cv2.merge((l, a, b)), cv2.COLOR_LAB2RGB)

    def crop_frame(img):
        h = img.shape[0]
        top = int(h * 0.35)
        bottom = int(h * 0.90)
        return img[top:bottom, :, :]

    def preprocess_for_cnn(img):
        resized = cv2.resize(img, (200, 66), interpolation=cv2.INTER_AREA)
        yuv = cv2.cvtColor(resized, cv2.COLOR_RGB2YUV)
        return yuv.astype(np.float32) / 255.0

# Configuration
IMG_HEIGHT = 66
IMG_WIDTH = 200
IMG_CHANNELS = 3
BATCH_SIZE = 64
EPOCHS = 100
LEARNING_RATE = 1e-4
VALIDATION_SPLIT = 0.2

BASE_DIR = PROJECT_ROOT if PROJECT_ROOT is not None else Path.cwd().resolve()
MODEL_DIR = BASE_DIR / 'models'
MODEL_DIR.mkdir(parents=True, exist_ok=True)
MODEL_SAVE_PATH = str(MODEL_DIR / 'traffiq1_v2.h5')
TFLITE_SAVE_PATH = str(MODEL_DIR / 'traffiq1_v2.tflite')

SPEED_LOSS_WEIGHT = 0.4
DIRECTION_LOSS_WEIGHT = 0.6

AUTO_DATASET_DIR = _find_dataset_dir(PROJECT_ROOT)

# Optional manual override if needed
MANUAL_DATA_DIR = os.environ.get('DATA_DIR', '').strip()

DATA_DIR = None
if MANUAL_DATA_DIR:
    manual_path = Path(MANUAL_DATA_DIR)
    if _is_valid_dataset_dir(manual_path):
        DATA_DIR = str(manual_path)
    else:
        print(f"[Setup] Ignoring invalid manual DATA_DIR: {MANUAL_DATA_DIR}")

if DATA_DIR is None:
    if AUTO_DATASET_DIR is not None:
        DATA_DIR = str(AUTO_DATASET_DIR)
    else:
        preferred = [
            Path('/content/drive/MyDrive/dataset_car'),
            Path('/content/drive/MyDrive/dataset'),
            Path('/content/dataset_car'),
            Path('/content/dataset'),
        ]
        for p in preferred:
            if _is_valid_dataset_dir(p):
                DATA_DIR = str(p)
                break

if DATA_DIR is None:
    DATA_DIR = '/content/drive/MyDrive/dataset_car'

print('PROJECT_ROOT =', PROJECT_ROOT)
print('MODEL_DIR =', MODEL_DIR)
print('AUTO_DATASET_DIR =', AUTO_DATASET_DIR)
print('DATA_DIR =', DATA_DIR)
if AUTO_DATASET_DIR is None and not _is_valid_dataset_dir(Path(DATA_DIR)):
    print('[Setup] Could not auto-find a valid dataset. Run Cell 3, then set DATA_DIR manually.')
if os.environ.get('ENABLE_DEEP_DATASET_SEARCH', '0') != '1':
    print("[Setup] Deep dataset recursive search is OFF (fast mode). Set ENABLE_DEEP_DATASET_SEARCH=1 only if needed.")

In [12]:
def augment_brightness(image: np.ndarray) -> np.ndarray:
    hsv = cv2.cvtColor(image, cv2.COLOR_RGB2HSV).astype(np.float32)
    factor = 0.4 + np.random.uniform()
    hsv[:, :, 2] = np.clip(hsv[:, :, 2] * factor, 0, 255)
    return cv2.cvtColor(hsv.astype(np.uint8), cv2.COLOR_HSV2RGB)


def augment_lighting_color(image: np.ndarray) -> np.ndarray:
    hsv = cv2.cvtColor(image, cv2.COLOR_RGB2HSV).astype(np.float32)
    hue_shift = np.random.uniform(-15, 15)
    hsv[:, :, 0] = (hsv[:, :, 0] + hue_shift) % 180
    sat_factor = 0.8 + np.random.uniform() * 0.4
    hsv[:, :, 1] = np.clip(hsv[:, :, 1] * sat_factor, 0, 255)
    return cv2.cvtColor(hsv.astype(np.uint8), cv2.COLOR_HSV2RGB)


def augment_shadow(image: np.ndarray) -> np.ndarray:
    h, w = image.shape[:2]
    x1, x2 = np.random.randint(0, w, 2)
    shadow_mask = np.zeros_like(image[:, :, 0])
    pts = np.array([[x1, 0], [x2, h], [w, h], [w, 0]], dtype=np.int32)
    cv2.fillPoly(shadow_mask, [pts], 1)
    image = image.copy().astype(np.float32)
    image[shadow_mask == 1] *= 0.5
    return np.clip(image, 0, 255).astype(np.uint8)


def augment_flip(image: np.ndarray, direction: float):
    return cv2.flip(image, 1), -direction

In [13]:
def load_donkey_catalog(data_dir: str) -> list:
    records = []
    sub_dirs = sorted(os.listdir(data_dir))

    for sub in sub_dirs:
        sub_path = os.path.join(data_dir, sub)
        if not os.path.isdir(sub_path):
            continue

        catalog_files = sorted(glob.glob(os.path.join(sub_path, 'catalog_*.catalog')))
        if not catalog_files:
            continue

        img_dir = os.path.join(sub_path, 'images')
        loaded = 0

        for cat_file in catalog_files:
            with open(cat_file) as f:
                for line in f:
                    line = line.strip()
                    if not line:
                        continue
                    try:
                        rec = json.loads(line)
                    except json.JSONDecodeError:
                        continue

                    if 'user/angle' not in rec or 'cam/image_array' not in rec:
                        continue

                    img_path = os.path.join(img_dir, rec['cam/image_array'])
                    records.append({
                        'steering': float(rec['user/angle']),
                        'throttle': float(rec.get('user/throttle', 0.3)),
                        'image_path_abs': img_path,
                        'source': sub,
                    })
                    loaded += 1

        if loaded > 0:
            print(f'  [Loaded] {sub:40s} -> {loaded:6d} records')

    return records


def load_labels_json(data_dir: str) -> list:
    label_file = Path(data_dir) / 'labels.json'
    with open(label_file) as f:
        raw_records = json.load(f)

    records = []
    for r in raw_records:
        img_path = str(Path(data_dir) / 'images' / os.path.basename(r.get('image_path', '')))
        records.append({
            'steering': float(r.get('steering', r.get('user/angle', 0.0))),
            'throttle': float(r.get('throttle', r.get('user/throttle', 0.3))),
            'image_path_abs': img_path,
            'source': 'labels_json',
        })
    return records


def load_dataset(data_dir: str) -> list:
    label_file = Path(data_dir) / 'labels.json'
    if label_file.exists():
        print('[Dataset] Detected TRAFFIQ format (labels.json)')
        return load_labels_json(data_dir)

    print('[Dataset] Detected Donkey Car catalog format')
    records = load_donkey_catalog(data_dir)
    if not records:
        raise FileNotFoundError(f'No labels.json or catalog_*.catalog files found in {data_dir}')
    return records

In [14]:
class TraffiqDatasetV2(tf.keras.utils.Sequence):
    def __init__(self, records: list, batch_size: int, augment: bool = False, **kwargs):
        super().__init__(**kwargs)
        self.records = records
        self.batch_size = batch_size
        self.augment = augment

    def __len__(self):
        return len(self.records) // self.batch_size

    def __getitem__(self, idx):
        batch = self.records[idx * self.batch_size:(idx + 1) * self.batch_size]
        images, labels = [], []

        for rec in batch:
            img = cv2.imread(rec['image_path_abs'])
            if img is None:
                continue

            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            direction = float(rec['steering'])
            speed = float(rec['throttle'])

            if self.augment:
                img = augment_brightness(img)
                img = augment_lighting_color(img)
                if np.random.random() > 0.5:
                    img = augment_shadow(img)
                if np.random.random() > 0.5:
                    img, direction = augment_flip(img, direction)

            normalized = normalize_lighting(img)
            cropped = crop_frame(normalized)
            processed = preprocess_for_cnn(cropped)

            images.append(processed)
            labels.append([speed, direction])

        return np.array(images), np.array(labels, dtype=np.float32)

    def on_epoch_end(self):
        np.random.shuffle(self.records)


def build_traffiq_v2_model():
    inputs = tf.keras.Input(shape=(IMG_HEIGHT, IMG_WIDTH, IMG_CHANNELS), name='image_input')

    x = layers.Conv2D(24, (5, 5), strides=(2, 2), activation='elu', name='conv1')(inputs)
    x = layers.Conv2D(36, (5, 5), strides=(2, 2), activation='elu', name='conv2')(x)
    x = layers.Conv2D(48, (5, 5), strides=(2, 2), activation='elu', name='conv3')(x)
    x = layers.Conv2D(64, (3, 3), activation='elu', name='conv4')(x)
    x = layers.Conv2D(64, (3, 3), activation='elu', name='conv5')(x)
    x = layers.Dropout(0.3, name='dropout1')(x)
    x = layers.Flatten(name='flatten')(x)

    x = layers.Dense(100, activation='elu', name='dense1')(x)
    x = layers.Dropout(0.3, name='dropout2')(x)
    x = layers.Dense(50, activation='elu', name='dense2')(x)

    dir_x = layers.Dense(10, activation='elu', name='dir_dense')(x)
    direction = layers.Dense(1, activation='tanh', name='direction')(dir_x)

    spd_x = layers.Dense(10, activation='elu', name='spd_dense')(x)
    speed = layers.Dense(1, activation='tanh', name='speed')(spd_x)

    output = layers.Concatenate(name='output')([speed, direction])

    return tf.keras.Model(inputs=inputs, outputs=output, name='TRAFFIQ_v2')

In [15]:
def plot_training_curves(history):
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle('TRAFFIQ v2 - Training Results [Speed, Direction]', fontsize=14, fontweight='bold')

    axes[0].plot(history.history['loss'], label='Train Loss', color='#4A90D9')
    axes[0].plot(history.history['val_loss'], label='Val Loss', color='#E74C3C')
    axes[0].set_title('Weighted Loss (total)')
    axes[0].set_xlabel('Epoch')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    axes[1].plot(history.history['speed_mae'], label='Train', color='#2ECC71')
    axes[1].plot(history.history['val_speed_mae'], label='Val', color='#27AE60')
    axes[1].set_title('Speed MAE')
    axes[1].set_xlabel('Epoch')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    axes[1].axhline(0.08, color='red', linestyle='--', linewidth=1)

    axes[2].plot(history.history['direction_mae'], label='Train', color='#F39C12')
    axes[2].plot(history.history['val_direction_mae'], label='Val', color='#E67E22')
    axes[2].set_title('Direction MAE')
    axes[2].set_xlabel('Epoch')
    axes[2].legend()
    axes[2].grid(True, alpha=0.3)
    axes[2].axhline(0.05, color='red', linestyle='--', linewidth=1)

    plt.tight_layout()
    curve_path = MODEL_DIR / 'training_curves_v2.png'
    plt.savefig(curve_path, dpi=150)
    print('[Saved] Training curves ->', curve_path)


def export_tflite(model, data_dir: str):
    all_records = load_dataset(data_dir)
    sample = random.sample(all_records, min(300, len(all_records)))

    def representative_dataset():
        for rec in sample:
            img = cv2.imread(rec['image_path_abs'])
            if img is None:
                continue
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            normalized = normalize_lighting(img)
            cropped = crop_frame(normalized)
            processed = preprocess_for_cnn(cropped)
            yield [processed[np.newaxis, ...]]

    converter = tf.lite.TFLiteConverter.from_keras_model(model)
    converter.optimizations = [tf.lite.Optimize.DEFAULT]
    converter.representative_dataset = representative_dataset
    converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
    converter.inference_input_type = tf.int8
    converter.inference_output_type = tf.int8

    tflite_model = converter.convert()
    with open(TFLITE_SAVE_PATH, 'wb') as f:
        f.write(tflite_model)

    size_kb = os.path.getsize(TFLITE_SAVE_PATH) / 1024
    print(f'[TFLite INT8] Saved -> {TFLITE_SAVE_PATH} ({size_kb:.1f} KB)')


def benchmark_inference(model, n_runs: int = 100):
    import time

    dummy = np.random.rand(1, IMG_HEIGHT, IMG_WIDTH, IMG_CHANNELS).astype(np.float32)
    times = []
    for _ in range(n_runs):
        t0 = time.perf_counter()
        model.predict(dummy, verbose=0)
        times.append((time.perf_counter() - t0) * 1000)

    avg = np.mean(times[10:])
    est_pi = avg * 3.5
    status = 'OK' if est_pi < 80 else 'Too slow'
    print(f'[Benchmark] Avg inference (PC): {avg:.1f} ms')
    print(f'[Benchmark] Est. on Pi 4B: {est_pi:.0f} ms ({status})')

In [ ]:
SELECTED_DATASETS = [
    'american_steel_adam_2',
    'athena_rainer_bosch',
    'circuit_launch_ed_2',
]
USE_HALF_DATA = True

# Fast iteration controls
FAST_MODE = False
FAST_EPOCHS = 20
FAST_BATCH_SIZE = 128
FAST_PATIENCE = 5
FAST_LR_PATIENCE = 2

# v3 direction-focused tuning
SPEED_LOSS_WEIGHT = 0.2
DIRECTION_LOSS_WEIGHT = 0.8
STEERING_CLIP = 1.0
TURN_THRESHOLD = 0.08
HARD_TURN_THRESHOLD = 0.20
HARD_TURN_OVERSAMPLE = 2
DIRECTION_HUBER_DELTA = 0.10
DIRECTION_MAE_BLEND = 0.35


def clip_controls(speed: float, direction: float):
    speed = float(np.clip(speed, -1.0, 1.0))
    direction = float(np.clip(direction, -STEERING_CLIP, STEERING_CLIP))
    return speed, direction


def rebalance_direction_records(records):
    straight = [r for r in records if abs(r['steering']) < TURN_THRESHOLD]
    turning = [r for r in records if abs(r['steering']) >= TURN_THRESHOLD]
    hard_turns = [r for r in turning if abs(r['steering']) >= HARD_TURN_THRESHOLD]

    balanced = turning + straight[:len(turning)]
    if hard_turns and HARD_TURN_OVERSAMPLE > 0:
        balanced.extend(random.choices(hard_turns, k=len(hard_turns) * HARD_TURN_OVERSAMPLE))

    np.random.shuffle(balanced)
    return balanced


def _process_record_numpy(image_path_abs, steering, throttle, augment):
    image_path = image_path_abs.decode('utf-8') if isinstance(image_path_abs, (bytes, bytearray)) else str(image_path_abs)
    img = cv2.imread(image_path)
    if img is None:
        x = np.zeros((IMG_HEIGHT, IMG_WIDTH, IMG_CHANNELS), dtype=np.float32)
        y = np.array([0.0, 0.0], dtype=np.float32)
        return x, y

    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    direction = float(steering)
    speed = float(throttle)
    speed, direction = clip_controls(speed, direction)

    if augment:
        img = augment_brightness(img)
        img = augment_lighting_color(img)
        if np.random.random() > 0.5:
            img = augment_shadow(img)
        if np.random.random() > 0.5:
            img, direction = augment_flip(img, direction)
            speed, direction = clip_controls(speed, direction)

    cropped = crop_frame(img)
    processed = preprocess_for_cnn(cropped).astype(np.float32)
    y = np.array([speed, direction], dtype=np.float32)
    return processed, y


def _make_tf_dataset(records, batch_size, training=True):
    image_paths = [r['image_path_abs'] for r in records]
    steerings = [float(r['steering']) for r in records]
    throttles = [float(r['throttle']) for r in records]

    ds = tf.data.Dataset.from_tensor_slices((image_paths, steerings, throttles))
    if training:
        ds = ds.shuffle(buffer_size=min(len(records), 10000), reshuffle_each_iteration=True)

    def _map_fn(path, steering, throttle):
        x, y = tf.numpy_function(
            func=lambda p, s, t: _process_record_numpy(p, s, t, training),
            inp=[path, steering, throttle],
            Tout=(tf.float32, tf.float32),
        )
        x.set_shape((IMG_HEIGHT, IMG_WIDTH, IMG_CHANNELS))
        y.set_shape((2,))
        return x, y

    ds = ds.map(_map_fn, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(batch_size, drop_remainder=True)
    ds = ds.prefetch(tf.data.AUTOTUNE)
    return ds


def train(data_dir: str, epochs: int = EPOCHS):
    data_path = Path(data_dir)
    if not data_path.exists():
        hint_paths = [
            '/content/dataset_car',
            '/content/dataset',
            '/content/drive/MyDrive/dataset_car',
            '/content/drive/MyDrive/dataset',
            '/content/drive/MyDrive/Trafiic_car_autonomous/traffiq/dataset_car',
            '/content/drive/MyDrive/Trafiic_car_autonomous/traffiq/dataset',
            '/content/drive/MyDrive/traffiq/dataset_car',
            '/content/drive/MyDrive/traffiq/dataset',
        ]
        msg = [f'[Error] DATA_DIR not found: {data_dir}', '[Hint] Set DATA_DIR to one of these if it exists:'] + hint_paths
        raise FileNotFoundError('\n'.join(msg))

    records = load_dataset(str(data_path))
    print(f'\n[Dataset] Loaded {len(records)} records from {data_path}')

    selected_set = set(SELECTED_DATASETS)
    selected_records = [r for r in records if r.get('source') in selected_set]
    if selected_records:
        print(f'[Dataset] Selected sources: {len(selected_records)} records from {len(selected_set)} requested sub-datasets')
        records = selected_records
    else:
        print('[Dataset] Warning: selected sub-datasets not found in loaded records; using all loaded records.')

    if USE_HALF_DATA:
        by_source = {}
        for r in records:
            src = r.get('source', 'unknown')
            by_source.setdefault(src, []).append(r)

        half_records = []
        print('[Dataset] Using 50% per source:')
        for src in sorted(by_source.keys()):
            recs = by_source[src]
            keep_n = max(1, len(recs) // 2)
            sampled = random.sample(recs, keep_n) if len(recs) > keep_n else recs
            half_records.extend(sampled)
            print(f'  - {src:40s}: {len(sampled):6d} / {len(recs):6d}')

        records = half_records
        print(f'[Dataset] After half-data sampling: {len(records)} records')

    all_angles = [r['steering'] for r in records]
    print(f'[Dataset] Steering range: [{min(all_angles):.3f}, {max(all_angles):.3f}]')
    print(f'[Dataset] Unique angles: {len(set(round(a, 4) for a in all_angles))}')

    balanced = rebalance_direction_records(records)
    straight = [r for r in balanced if abs(r['steering']) < TURN_THRESHOLD]
    turning = [r for r in balanced if abs(r['steering']) >= TURN_THRESHOLD]
    hard_turns = [r for r in balanced if abs(r['steering']) >= HARD_TURN_THRESHOLD]
    print(
        f'[Dataset] Balanced(v3): {len(balanced)} records '
        f'({len(turning)} turning, {len(straight)} straight, {len(hard_turns)} hard-turn samples)'
    )

    train_recs, val_recs = train_test_split(balanced, test_size=VALIDATION_SPLIT, random_state=42)

    run_epochs = FAST_EPOCHS if FAST_MODE else max(epochs, 120)
    run_batch_size = FAST_BATCH_SIZE if FAST_MODE else BATCH_SIZE
    es_patience = FAST_PATIENCE if FAST_MODE else 10
    rlr_patience = FAST_LR_PATIENCE if FAST_MODE else 3

    print(f'[TrainConfig] FAST_MODE={FAST_MODE} | batch_size={run_batch_size} | epochs={run_epochs}')

    train_ds = _make_tf_dataset(train_recs, batch_size=run_batch_size, training=True)
    val_ds = _make_tf_dataset(val_recs, batch_size=run_batch_size, training=False)

    model = build_traffiq_v2_model()
    model.summary()

    def weighted_mse(y_true, y_pred):
        speed_loss = tf.reduce_mean(tf.square(y_true[:, 0] - y_pred[:, 0]))
        direction_huber = tf.reduce_mean(
            tf.keras.losses.huber(y_true[:, 1], y_pred[:, 1], delta=DIRECTION_HUBER_DELTA)
        )
        direction_mae_raw = tf.reduce_mean(tf.abs(y_true[:, 1] - y_pred[:, 1]))
        direction_loss = (1.0 - DIRECTION_MAE_BLEND) * direction_huber + DIRECTION_MAE_BLEND * direction_mae_raw
        return SPEED_LOSS_WEIGHT * speed_loss + DIRECTION_LOSS_WEIGHT * direction_loss

    def speed_mae(y_true, y_pred):
        return tf.reduce_mean(tf.abs(y_true[:, 0] - y_pred[:, 0]))

    def direction_mae(y_true, y_pred):
        return tf.reduce_mean(tf.abs(y_true[:, 1] - y_pred[:, 1]))

    model.compile(
        optimizer=optimizers.Adam(learning_rate=LEARNING_RATE),
        loss=weighted_mse,
        metrics=[speed_mae, direction_mae],
    )

    cb_list = [
        callbacks.ModelCheckpoint(MODEL_SAVE_PATH, monitor='val_loss', save_best_only=True, verbose=1),
        callbacks.EarlyStopping(monitor='val_loss', patience=es_patience, restore_best_weights=True, verbose=1),
        callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=rlr_patience, min_lr=1e-6, verbose=1),
        callbacks.CSVLogger(str(MODEL_DIR / 'training_log_v2.csv')),
    ]

    print(f'\n[Training] Starting for up to {run_epochs} epochs...\n')
    history = model.fit(train_ds, validation_data=val_ds, epochs=run_epochs, callbacks=cb_list)

    plot_training_curves(history)
    export_tflite(model, str(data_path))
    benchmark_inference(model)
    return model


# Run training
MODEL = train(DATA_DIR, EPOCHS)